## Q1. Prepare the dataset

Start with `baseline_model_nyc_taxi_data.ipynb`. Download the March 2024 Green Taxi data. We will use this data to simulate a production usage of a taxi trip duration prediction service.

What is the shape of the downloaded data? How many rows are there?

* 72044
* 78537 
* 57457
* 54396

In [2]:
import datetime
import pandas as pd

from joblib import dump, load

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [3]:
year = 2024
month = 3

input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year:04d}-{month:02d}.parquet'

df = pd.read_parquet(input_file)
print(f"✅ DataFrame shape: {df.shape}")
print(f"✅ DataFrame rows: {df.shape[0]}")

✅ DataFrame shape: (57457, 20)
✅ DataFrame rows: 57457


## Q2. Metric

Let's expand the number of data quality metrics we’d like to monitor! Please add one metric of your choice and a quantile value for the `"fare_amount"` column (`quantile=0.5`).

Hint: explore evidently metric `ColumnQuantileMetric` (from `evidently.metrics import ColumnQuantileMetric`) 

What metric did you choose?

In [4]:
from evidently import Dataset
from evidently import DataDefinition
from evidently.metrics import QuantileValue
from evidently import Report
from evidently.metrics import MeanValue, MaxValue

In [5]:
# Build a dataset with fare_amount as the numerical column to monitor.
data_definition = DataDefinition(numerical_columns=["fare_amount"])
dataset = Dataset.from_pandas(
    df[["fare_amount"]],
    data_definition=data_definition,
)

# MeanValue is the additional data-quality metric; QuantileValue reports the median.
report = Report(metrics=[
    MeanValue(column="fare_amount"),
    QuantileValue(column="fare_amount", quantile=0.5),
])


## Q3. Monitoring

Let’s start monitoring. Run expanded monitoring for a new batch of data (March 2024). 

What is the maximum value of metric `quantile = 0.5` on the `"fare_amount"` column during March 2024 (calculated daily)?

* 10
* 12.5
* 14.2
* 14.8

In [ ]:
result = report.run(current_data=dataset, reference_data=None)
result

In [9]:
import pandas as pd
from evidently import Report
from evidently.metrics import QuantileValue

march_data = df.copy()
pickup_datetime = pd.to_datetime(march_data["lpep_pickup_datetime"])

march_data = march_data.loc[
    (pickup_datetime >= "2024-03-01")
    & (pickup_datetime < "2024-04-01")
].copy()
march_data["pickup_date"] = pickup_datetime.loc[march_data.index].dt.date

median_report = Report(
    metrics=[QuantileValue(column="fare_amount", quantile=0.5)]
)

daily_medians = {}
for date, day_data in march_data.groupby("pickup_date"):
    fare_data = day_data[["fare_amount"]].dropna()
    if fare_data.empty:
        continue

    snapshot = median_report.run(current_data=fare_data, reference_data=None)
    metric = next(
        item for item in snapshot.dict()["metrics"]
        if item["metric_name"] == "QuantileValue(column=fare_amount,quantile=0.5)"
    )
    daily_medians[date] = float(metric["value"])

if not daily_medians:
    raise ValueError("No valid fare_amount values found for March 2024")

print(f"✅ Maximum daily fare_amount median: {max(daily_medians.values())}")

✅ Maximum daily fare_amount median: 14.2


## Q4. Dashboard


Finally, let’s add panels with new added metrics to the dashboard. After we customize the  dashboard let's save a dashboard config, so that we can access it later. Hint: click on “Save dashboard” to access JSON configuration of the dashboard. This configuration should be saved locally.

Where to place a dashboard config file?

* `project_folder` (05-monitoring)
* `project_folder/config`  (05-monitoring/config)
* `project_folder/dashboards`  (05-monitoring/dashboards)
* `project_folder/data`  (05-monitoring/data)

In [12]:
from evidently.ui.workspace import Workspace
from evidently.ui.workspace import WorkspaceBase

# Create a workspace
ws = Workspace.create("workspace")

# Create project
project = ws.create_project("NYC Taxi Data Quality Project")
project.description = "Quality control for NYC Taxi data using Evidently AI"
project.save()

project

Project ID: 01a105a9-bae4-7ed7-bacd-26874bb086bb
Project Name: NYC Taxi Data Quality Project
Project Description: Quality control for NYC Taxi data using Evidently AI
        

In [ ]:
# Add the report snapshot to the project so its metrics appear in the dashboard.
run_name = "March 2024 fare amount metrics"
run = None
for run_id in ws.list_runs(project.id):
    saved_run = ws.get_run(project.id, run_id)
    if saved_run is not None and saved_run.name == run_name:
        run = saved_run
        break

if run is None:
    run = ws.add_run(
        project.id,
        result,
        include_data=True,
        name=run_name,
    )

from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import counter_panel

dashboard = project.dashboard
dashboard_model = dashboard.model()
for tab in dashboard_model.tabs:
    tab.panels = list(dict.fromkeys(tab.panels))
ws.save_dashboard(project.id, dashboard_model)
if not dashboard_model.tabs:
    dashboard.add_tab("Fare amount metrics")

existing_panel_titles = {panel.title for panel in dashboard.model().panels}
metric_panels = [
    counter_panel(
        title="Mean fare amount",
        values=[
            PanelMetric(
                legend="Mean fare",
                metric="evidently:metric_v2:MeanValue",
                metric_labels={"column": "fare_amount"},
            )
        ],
        size="half",
        aggregation="last",
    ),
    counter_panel(
        title="Median fare amount (q=0.5)",
        values=[
            PanelMetric(
                legend="Median fare",
                metric="evidently:metric_v2:QuantileValue",
                metric_labels={"column": "fare_amount", "quantile": "0.5"},
            )
        ],
        size="half",
        aggregation="last",
    ),
]

for panel in metric_panels:
    if panel.title not in existing_panel_titles:
        dashboard.add_panel(panel)
        existing_panel_titles.add(panel.title)

project.save()
run

In [14]:
import shutil
from pathlib import Path

# Copy the dashboard JSON saved by Evidently UI into the project dashboards folder.
exported_config = Path("workspace") / str(project.id) / "dashboard.json"
dashboards_dir = Path("dashboards")
dashboards_dir.mkdir(parents=True, exist_ok=True)

if not exported_config.is_file():
    raise FileNotFoundError(f"Evidently dashboard export not found: {exported_config}")

saved_config = dashboards_dir / "dashboard.json"
shutil.copy2(exported_config, saved_config)

print(f"✅ Dashboard configuration saved to {saved_config}")

✅ Dashboard configuration saved to dashboards/dashboard.json
